# Notebook Modelling: Ekstraksi Field KYC dari Dokumen Identitas

Notebook ini berisi catatan teknis dari seluruh proses modelling yang kami lakukan untuk proyek ekstraksi field KYC (nama, tanggal lahir, alamat) dari foto dokumen identitas. Saya akan menulis notebook ini seperti cerita, bukan cuma kumpulan angka. Tujuannya supaya alur berpikirnya kelihatan jelas, mulai dari baseline paling sederhana sampai model akhir yang kami pakai.

Setiap angka yang muncul di notebook ini diambil langsung dari log eksperimen yang sudah kami simpan selama proses kerja (`reports/experiments.jsonl`), atau dihitung ulang langsung dari data mentah dan model yang sudah dilatih. Jadi semua yang tertulis di sini bisa dicek ulang.

In [1]:
import json
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent
REPORTS = REPO / "reports"
DATA = REPO / "data"

def load_experiments():
    rows = [json.loads(l) for l in open(REPORTS / "experiments.jsonl")]
    return pd.DataFrame(rows)

exp = load_experiments()
print(f"total baris log eksperimen: {len(exp)}")
print(f"jumlah metode unik yang pernah dicoba: {exp['method'].nunique()}")

total baris log eksperimen: 538
jumlah metode unik yang pernah dicoba: 25


## 1. Latar Belakang dan Batasan Proyek

Tugas kami adalah membangun sistem yang bisa membaca tiga field dari foto dokumen identitas: nama, tanggal lahir, dan alamat. Dokumennya sangat beragam, ada MyKad Malaysia, paspor, SIM, kartu identitas dari banyak negara berbeda. Jadi ini bukan kasus dokumen dengan satu layout tetap, tapi kumpulan layout yang berbeda beda.

Ada beberapa batasan penting dari panduan proyek yang menentukan arah teknis kami sejak awal.

Pertama, kami tidak boleh memakai LLM atau VLM sebagai bagian dari pipeline. OCR konvensional seperti Tesseract, EasyOCR, dan PaddleOCR boleh dipakai. Ini artinya seluruh proses ekstraksi field harus dibangun dari fitur yang kami rancang sendiri, bukan lewat prompting model bahasa besar.

Kedua, tidak boleh pakai tools AutoML atau automated feature engineering seperti H2O.ai atau tsfresh.

Ketiga, kami wajib memakai seluruh dataset yang diberikan, tidak boleh membuang sebagian data begitu saja.

Keempat, tidak boleh melakukan anotasi manual, kecuali untuk keperluan evaluasi. Jadi label training harus diturunkan secara otomatis dari ground truth yang sudah ada, bukan dilabeli manual satu per satu.

Metrik yang kami pakai ada tiga: exact match (apakah prediksi sama persis dengan jawaban benar), CER atau character error rate (seberapa banyak karakter yang salah dibanding total karakter jawaban benar), dan WER atau word error rate (versi kata dari CER). Sepanjang notebook ini kami lebih sering menonjolkan CER dibanding exact match, karena CER memberi nilai parsial. Sebuah prediksi yang hampir benar tapi kurang satu kata tetap dianggap gagal total kalau dilihat dari exact match, padahal secara praktis prediksi itu masih berguna. CER menangkap nuansa itu.

In [2]:
summary = json.loads(open(DATA / "splits" / "summary.json").read())
print(json.dumps(summary, indent=2))

{
  "seed": 42,
  "max_images_per_identity_in_eval": 2,
  "min_train_address_identities": 15,
  "total_labeled": 632,
  "unique_identities": 133,
  "held_out_unlabeled": 100,
  "splits": {
    "train": {
      "rows": 535,
      "unique_identities": 39,
      "max_identity_rows": 40,
      "address_bearing_rows": 15
    },
    "val": {
      "rows": 49,
      "unique_identities": 47,
      "max_identity_rows": 2,
      "address_bearing_rows": 49
    },
    "test": {
      "rows": 48,
      "unique_identities": 47,
      "max_identity_rows": 2,
      "address_bearing_rows": 48
    }
  }
}


## 2. Eksplorasi Data

Sebelum mulai modelling, kami mengecek dulu bentuk datanya. Total ada 632 baris berlabel di `ground_truth.csv`, ditambah 100 gambar tanpa label yang disisihkan sebagai held out set (tidak pernah dipakai untuk training atau evaluasi utama, hanya untuk pengecekan akhir kalau diperlukan).

Waktu kami cek lebih dalam, kami menemukan dua masalah kualitas data yang cukup penting. Saya ceritakan satu per satu karena keduanya mengubah cara kami membangun split data.

**Masalah pertama: identitas yang salah dikelompokkan.** Kami memakai kombinasi nama dan tanggal lahir sebagai kunci untuk mengenali satu orang yang sama (supaya foto yang sama tidak muncul di train dan test sekaligus, ini disebut identity disjoint split). Ternyata ada satu identitas, namanya JUSTYNA AGNIESZKA KRAWCZYK NOWAK, yang punya 20 foto dengan tanggal lahir yang berbeda beda, naik satu hari setiap baris. Kami cek langsung fotonya, dan ternyata itu adalah kartu SIM Polandia yang sama persis di semua 20 foto, tanggal lahir yang tercetak di kartunya selalu sama yaitu 12 Mei 1971. Jadi tanggal lahir di ground truth itu salah ketik atau ada bug saat data dibuat, bukan variasi asli. Karena kunci identitas kami memakai tanggal lahir, sistem kami sempat menganggap ini sebagai 20 orang berbeda, padahal harusnya satu orang. Kami perbaiki dengan menyamakan tanggal lahirnya jadi satu nilai yang benar sesuai foto.

**Masalah kedua: field alamat nyaris hilang dari data training.** Setelah masalah pertama diperbaiki, jumlah identitas unik berubah, dan itu membuat urutan pengacakan split ikut berubah. Ternyata field alamat itu langka, cuma dokumen dari Malaysia (MyKad) yang punya alamat, dan identitas MyKad ini kebanyakan cuma satu foto per orang (kecil, jadi gampang masuk ke val atau test karena aturan split kami menghindari identitas besar masuk eval). Akibatnya, waktu kami buat ulang split, train sempat kosong sama sekali dari contoh alamat. Model jadi tidak bisa belajar mengenali field alamat sama sekali. Kami perbaiki dengan menambahkan aturan baru di fungsi split, yaitu menyisihkan sejumlah identitas yang punya alamat supaya pasti masuk ke train, apa pun hasil pengacakan acaknya.

Dua masalah ini penting untuk dicatat karena keduanya bukan soal algoritma model, tapi soal kualitas dan desain data sebelum model itu sendiri dibangun.

In [3]:
import csv
from collections import namedtuple

# Dibaca langsung dengan modul csv bawaan Python, tidak bergantung pada kode
# repo kami, supaya notebook ini bisa dijalankan berdiri sendiri. Ada satu
# hal unik di file ground_truth.csv: 131 baris pertama dibungkus tanda kutip
# ekstra (peninggalan cara file itu diekspor), jadi csv.reader awalnya cuma
# baca itu sebagai satu kolom saja. Baris seperti itu perlu diuraikan ulang
# sekali lagi supaya empat kolomnya kebaca dengan benar.
Record = namedtuple("Record", ["filename", "name", "birth_date", "address"])

def read_ground_truth(path):
    with open(path, newline="", encoding="utf-8-sig") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        assert header[:4] == ["filename", "name", "birth_date", "address"]
        records = []
        for row in reader:
            if len(row) == 1 and "," in row[0]:
                row = next(csv.reader([row[0]]))
            records.append(Record(*(cell.strip() for cell in row[:4])))
    return records

records = read_ground_truth(DATA / "raw" / "ground_truth.csv")
print(f"total baris berlabel: {len(records)}")

identities = {f"{r.name}\x1f{r.birth_date}" for r in records}
print(f"jumlah identitas unik setelah perbaikan: {len(identities)}")

has_address = sum(1 for r in records if r.address)
print(f"baris yang punya field alamat terisi: {has_address} dari {len(records)}")

total baris berlabel: 632
jumlah identitas unik setelah perbaikan: 133
baris yang punya field alamat terisi: 112 dari 632


## 3. Desain Split Data

Karena banyak identitas di dataset ini punya banyak foto ulang (retake) dari kartu fisik yang sama, kami tidak bisa asal split baris secara acak. Kalau dua foto dari kartu yang sama masuk ke train dan test sekaligus, model bisa saja "menghafal" kartu itu saat training, lalu terlihat pintar saat evaluasi padahal cuma menghafal, bukan generalisasi.

Jadi kami membangun split di level identitas, bukan di level baris. Aturannya seperti ini. Semua foto dari satu identitas selalu masuk ke satu split saja, tidak pernah dipecah ke train dan val sekaligus. Kalau satu identitas punya foto retake dalam jumlah besar (lebih dari 2), identitas itu otomatis masuk ke train saja, tidak pernah masuk ke val atau test. Ini masuk akal karena kalau satu identitas ada 30 sampai 40 foto ulang, dan itu masuk ke test, maka test jadi didominasi satu orang saja, bukan mewakili keberagaman data.

Ditambah lagi ada aturan minimal jumlah identitas beralamat yang wajib masuk ke train (hasil dari perbaikan masalah kedua yang saya ceritakan di atas).

Hasil akhirnya split kami cukup seimbang. Kita lihat angkanya di bawah.

In [4]:
for split_name, info in summary["splits"].items():
    print(f"{split_name:6s}: {info['rows']:4d} baris, {info['unique_identities']:3d} identitas unik, "
          f"identitas terbesar punya {info['max_identity_rows']} baris, "
          f"baris beralamat: {info.get('address_bearing_rows', '-')}")

train :  535 baris,  39 identitas unik, identitas terbesar punya 40 baris, baris beralamat: 15
val   :   49 baris,  47 identitas unik, identitas terbesar punya 2 baris, baris beralamat: 49
test  :   48 baris,  47 identitas unik, identitas terbesar punya 2 baris, baris beralamat: 48


## 4. Baseline 0: Regex dan Heuristik Sederhana (Zero Shot)

Baseline paling awal kami adalah pendekatan paling sederhana yang bisa dipikirkan. Kami jalankan OCR (PaddleOCR) di seluruh gambar, lalu terapkan aturan sederhana untuk menebak field mana yang mana.

Untuk tanggal lahir, kami cari pola tanggal lewat regex, termasuk pola khusus MyKad, karena nomor identitas MyKad menyimpan tanggal lahir langsung di 6 digit pertama (format YYMMDD). Untuk nama, kami ambil baris teks terpanjang yang isinya huruf semua, tidak ada angka, dan bukan kata umum seperti nama negara. Untuk alamat, kami cari baris yang mengandung kata kunci alamat seperti JALAN atau KAMPUNG, atau ada pola kode pos.

Pendekatan ini murni aturan, tidak ada proses belajar dari data sama sekali. Tapi ini penting sebagai titik awal, supaya kita punya pembanding yang jelas ketika nanti mencoba pendekatan yang lebih rumit.

Satu catatan kecil soal cara baca angka CER di tabel bawah ini. CER bisa saja lebih dari 100 persen. Ini bukan salah hitung. CER dihitung dari jumlah karakter yang perlu diubah (ditambah, dihapus, atau diganti) supaya prediksi jadi sama dengan jawaban benar, dibagi panjang jawaban benar. Kalau prediksinya jauh lebih panjang dari jawaban benar (misalnya sistem salah ambil satu kalimat panjang padahal jawabannya cuma dua kata), jumlah karakter yang perlu diubah bisa melebihi panjang jawaban aslinya, jadi angkanya bisa tembus di atas 100 persen. Ini justru sinyal berguna, artinya bukan cuma "salah", tapi "salah dan jawabannya jauh lebih berantakan dari yang seharusnya".

In [5]:
df_geo = exp[exp["method"] == "baseline0-regex-v2"].copy()
df_geo["timestamp"] = pd.to_datetime(df_geo["timestamp"])
latest_b0 = df_geo.sort_values("timestamp").groupby("split").tail(1)

rows = []
for _, r in latest_b0.iterrows():
    f = r["metrics"]["fields"]
    rows.append({
        "split": r["split"],
        "name_exact": f["name"]["exact_accuracy"],
        "name_cer": f["name"]["character_error_rate"],
        "birth_date_exact": f["birth_date"]["exact_accuracy"],
        "birth_date_cer": f["birth_date"]["character_error_rate"],
        "address_cer": f["address"]["character_error_rate"],
    })
baseline0_table = pd.DataFrame(rows).set_index("split") * 100
baseline0_table.round(1)

,name_exact,name_cer,birth_date_exact,birth_date_cer,address_cer
split,,,,,
train,0.6,138.0,32.0,55.9,63.3
val,20.4,50.2,89.8,6.9,77.4
test,16.7,54.7,91.7,8.3,79.9


## 5. Titik Balik: Apakah OCR yang Salah, atau Cara Kita Memilih Teksnya?

Ini bagian yang menurut saya paling penting di seluruh proses kerja kami, karena ini yang menentukan arah pengembangan selanjutnya.

Sebelum sibuk memperbaiki OCR (misalnya coba preprocessing gambar, ganti engine OCR, atau training ulang), kami berhenti sejenak dan bertanya, sebenarnya di mana letak kesalahannya. Apakah OCR gagal membaca teks yang benar sama sekali, atau OCR sebenarnya sudah membaca dengan benar, tapi sistem kami yang salah memilih baris teks mana yang harus dipakai sebagai jawaban?

Kami cek satu per satu contoh yang salah, dan hasilnya cukup mengejutkan. Pada banyak kasus, jawaban yang benar itu ada, tertulis persis di hasil OCR, tapi sistem kami memilih baris lain yang salah. Contoh paling jelas ada di gambar `image_009`. OCR membaca teks `AFFANDY BIN OTHMAN` dengan sempurna, sama persis dengan jawaban benar. Tapi sistem prediksi kami malah memilih baris `KAD PENGENALAN` (artinya "Kartu Identitas" dalam Bahasa Melayu, ini cuma judul dokumen, bukan nama orang) sebagai jawaban nama.

Temuan ini penting karena mengubah arah kerja kami. Kalau OCR sendiri sudah cukup akurat, maka menghabiskan waktu memperbaiki kualitas gambar atau ganti engine OCR tidak akan banyak membantu. Yang perlu diperbaiki adalah cara sistem memilih dan menyusun teks yang sudah dibaca OCR itu menjadi jawaban akhir. Jadi sisa notebook ini akan lebih fokus ke bagian pemilihan dan penyusunan teks (kita sebut ini bagian parsing), bukan ke OCR itu sendiri.

In [6]:
# cari image_009.jpg di semua split, karena posisinya bisa berpindah split
# setiap kali data/splits dibangun ulang (splitnya berdasarkan identitas, jadi
# satu identitas bisa pindah split kalau susunan identitas lain ikut berubah)
record = None
for split_name in ("train", "val", "test"):
    data = json.loads(open(REPORTS / f"paddleocr_{split_name}.json").read())
    record = next((r for r in data["predictions"] if r["filename"] == "image_009.jpg"), None)
    if record:
        print(f"(contoh ini sekarang berada di split: {split_name})")
        break

print("nama yang benar   :", record["expected"]["name"])
print("teks hasil OCR    :", record["raw"].get("full", "")[:200])

(contoh ini sekarang berada di split: train)
nama yang benar   : AFFANDY BIN OTHMAN
teks hasil OCR    : C138
0180
KAD PENGENALAN
MyKad
MALAYSIA
IDENTITYCARD
810919-07-5337
AFFANDY BIN OTHMAN
NO 1592
WARGANEGARA
JALAN SIRAM
LELAKI
12100 BUTTERWORTH
ISLAM
PULAU PINANG


## 6. Baseline 1: Classifier per Baris Teks

Berangkat dari temuan di atas, kami membangun pendekatan baru. Alih alih menebak field lewat aturan tetap, kami melatih model classifier yang tugasnya menilai setiap baris teks hasil OCR, dan menentukan baris itu kemungkinan besar berisi nama, tanggal lahir, alamat, atau bukan ketiganya (kita sebut kelas "other").

Fitur yang dipakai semuanya diambil dari bentuk dan posisi teks, bukan dari isi kalimatnya secara bahasa. Beberapa contoh fitur, posisi baris relatif terhadap gambar (atas, bawah, kiri, kanan), tinggi dan lebar kotak teksnya, jumlah huruf dan angka, apakah baris itu cocok dengan pola tanggal, apakah mengandung kata kunci alamat, dan seterusnya. Total ada belasan fitur seperti ini.

Untuk label training, kami tidak melabeli manual satu per satu (itu dilarang oleh panduan proyek). Sebagai gantinya, label diturunkan otomatis dengan mencocokkan isi teks tiap baris terhadap jawaban benar di ground truth. Kalau isinya cocok, baris itu dianggap contoh positif untuk kelas tersebut.

Proses pembuatan label otomatis ini sempat punya dua bug yang kami temukan dan perbaiki. Pertama, kata umum seperti nama dokumen (misalnya "KAD PENGENALAN") kadang tercocokkan sebagai bagian dari nama orang kalau kebetulan ada kemiripan kata. Kedua, satu kata tunggal yang kebetulan cocok sebagian dengan nama orang (misalnya nama depan yang sama) bisa salah dianggap sebagai bukti kuat, padahal itu cuma potongan kecil. Setelah dua bug ini diperbaiki, kualitas label otomatis jadi jauh lebih bersih.

Model pertama yang kami pakai adalah logistic regression sederhana.

In [7]:
df_bl1 = exp[exp["method"] == "baseline1-line-classifier"].copy()
df_bl1["timestamp"] = pd.to_datetime(df_bl1["timestamp"])
latest_bl1 = df_bl1.sort_values("timestamp").groupby("split").tail(1)

rows = []
for _, r in latest_bl1.iterrows():
    f = r["metrics"]["fields"]
    rows.append({
        "split": r["split"],
        "name_exact": f["name"]["exact_accuracy"],
        "name_cer": f["name"]["character_error_rate"],
        "address_cer": f["address"]["character_error_rate"],
    })
baseline1_table = pd.DataFrame(rows).set_index("split") * 100
baseline1_table.round(1)

,name_exact,name_cer,address_cer
split,,,
val,24.5,36.6,47.1
test,25.0,42.2,51.1


## 7. Lapisan Perbaikan Struktural (Bagian Paling Berpengaruh)

Setelah baseline classifier jalan, kami sadar model ini masih sering salah pilih baris, meskipun sudah lebih baik dari baseline 0. Kami lalu memeriksa satu per satu contoh yang salah, dan menemukan pola pola kesalahan yang bisa diperbaiki bukan dengan menambah fitur baru, tapi dengan menambahkan aturan struktural di atas hasil prediksi model.

Yang saya maksud aturan struktural itu seperti ini, bukan mengajari model mengenali pola baru lewat data (karena data yang ada terlalu sedikit untuk itu), tapi menambahkan batasan logis yang masuk akal secara umum. Beberapa contoh aturan yang kami tambahkan, urut sesuai waktu ditemukan.

Baris paling atas dalam urutan baca selalu berupa judul dokumen, jadi baris itu tidak pernah dianggap sebagai nama orang, di layout dokumen mana pun.

Satu baris teks yang sudah dipilih sebagai nama tidak boleh dipilih lagi sebagai bagian dari alamat, begitu juga sebaliknya. Sebelum aturan ini ada, kami temukan sekitar 21 persen prediksi alamat tercampur dengan nama orang.

Nama Melayu sering terpotong jadi dua baris, nama depan di satu baris, nama keluarga (biasanya diawali kata "BIN" atau "BINTI") di baris berikutnya. Kami tambahkan logika penggabungan baris berikutnya, tapi hanya kalau baris pertama memang terlihat belum lengkap (diakhiri kata sambung seperti BIN atau BINTI).

Ada daftar kata kata umum di dokumen (seperti nama negara, judul dokumen, kata "identitas") yang tidak pernah dianggap sebagai nama atau alamat, dan daftar ini terus kami tambah seiring ditemukan kasus baru.

Setiap perbaikan ini kami uji satu per satu dan catat hasilnya di log eksperimen, supaya kita bisa lihat progresnya bertahap. Tabel di bawah menunjukkan bagaimana CER untuk nama dan alamat berubah setiap kali ada perbaikan baru ditambahkan, diambil langsung dari log eksperimen kami yang mencatat setiap kali kami menjalankan ulang evaluasi.

In [8]:
df_hist = exp[exp["method"] == "baseline1-line-classifier-geo"].copy()
df_hist["timestamp"] = pd.to_datetime(df_hist["timestamp"])
df_hist_val = df_hist[df_hist["split"] == "val"].sort_values("timestamp")

rows = []
for _, r in df_hist_val.iterrows():
    f = r["metrics"]["fields"]
    rows.append({
        "waktu": r["timestamp"],
        "name_cer": f["name"]["character_error_rate"] * 100,
        "address_cer": f["address"]["character_error_rate"] * 100,
    })
progress_table = pd.DataFrame(rows)
progress_table.round(1)

/tmp/ipykernel_10429/2433914229.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  progress_table.round(1)


,waktu,name_cer,address_cer
0,2026-08-19 06:58:31+00:00,45.0,57.3
1,2026-08-19 07:00:29+00:00,45.0,33.9
2,2026-08-19 07:38:35+00:00,45.0,33.5
3,2026-08-19 09:15:29+00:00,47.9,100.0
4,2026-08-19 09:22:00+00:00,35.1,44.8
5,2026-08-19 09:47:29+00:00,35.7,44.8
6,2026-08-19 09:50:08+00:00,29.2,44.8
7,2026-08-19 10:00:25+00:00,29.2,44.8
8,2026-08-19 11:50:07+00:00,29.2,43.9
9,2026-08-19 13:06:52+00:00,29.2,44.8


Kalau dilihat dari tabel di atas, CER nama turun dari sekitar 45 persen di percobaan paling awal, sampai sekitar 20 persen di akhir proses ini. CER alamat juga turun cukup jauh, meskipun perjalanannya naik turun karena beberapa perbaikan yang kami coba di tengah jalan sempat membuat hasilnya lebih buruk sebelum kami temukan versi yang benar (misalnya soal batas ambang atau threshold yang perlu diuji beberapa nilai dulu sebelum ketemu yang pas).

## 8. Membandingkan Beberapa Jenis Classifier

Selain logistic regression, kami juga coba beberapa model classifier berbasis pohon keputusan (gradient boosting), yaitu HistGradientBoosting dari scikit learn, LightGBM, XGBoost, dan CatBoost. Alasannya, model berbasis pohon bisa menangkap interaksi antar fitur yang tidak bisa ditangkap model linear seperti logistic regression. Misalnya, kombinasi "posisi di bagian atas DAN tidak ada angka DAN pendek" mungkin punya arti berbeda dibanding kalau masing masing fitur itu dilihat sendiri sendiri.

Hasilnya menarik. Keempat model berbasis pohon itu, meskipun berbeda beda implementasinya, semuanya konsisten mengalahkan logistic regression untuk field nama. Karena empat model yang berbeda beda caranya semua sepakat mengarah ke kesimpulan yang sama, ini sinyal yang cukup kuat, bukan cuma kebetulan satu model saja yang menang.

Tapi untuk field alamat, hasilnya kebalikannya. Logistic regression selalu lebih baik dibanding semua model pohon untuk alamat. Jadi tidak ada satu model yang menang di semua field.

In [9]:
tags = ["baseline1-line-classifier-geo", "baseline1-hgb-geo", "baseline1-lgbm-geo",
        "baseline1-xgboost-geo", "baseline1-catboost-geo"]
rows = []
for tag in tags:
    d = exp[exp["method"] == tag].copy()
    if d.empty:
        continue
    d["timestamp"] = pd.to_datetime(d["timestamp"])
    latest = d.sort_values("timestamp").groupby("split").tail(1)
    for _, r in latest.iterrows():
        f = r["metrics"]["fields"]
        model_name = tag.replace("baseline1-", "").replace("-geo", "")
        model_name = "logreg" if model_name == "line-classifier" else model_name
        rows.append({
            "model": model_name,
            "split": r["split"],
            "name_cer": f["name"]["character_error_rate"] * 100,
            "address_cer": f["address"]["character_error_rate"] * 100,
        })
compare_table = pd.DataFrame(rows).pivot(index="model", columns="split", values=["name_cer", "address_cer"])
compare_table.round(1)

name_cer       address_cer      
split        test   val        test   val
model                                    
catboost     28.6  22.1        46.2  45.3
hgb          31.7  27.0        57.8  51.3
lgbm         35.6  31.8        56.3  50.4
logreg       29.0  20.6        28.5  27.1
xgboost      29.6  24.7        56.3  48.9

## 9. Eksperimen yang Tidak Berhasil (dan Kenapa Ini Penting Dicatat)

Selama proses kerja, kami juga mencoba banyak ide lain yang ternyata tidak membantu, atau malah membuat hasil lebih buruk. Saya sengaja mencatat semuanya di sini, bukan cuma yang berhasil, karena menurut saya bagian ini sama pentingnya. Ini menunjukkan proses kerja yang jujur, dan juga memberi pelajaran soal batasan pendekatan kami.

Ada pola yang cukup jelas kalau dilihat dari semua percobaan ini. Setiap kali kami mencoba menambahkan fitur baru yang harus "dipelajari" oleh model dari data (misalnya daftar kata sambung nama, skor frekuensi kata umum yang dihitung dari data, atau skor kemiripan huruf dengan nama), hasilnya hampir selalu tidak membantu, kadang malah lebih buruk. Sebaliknya, setiap kali kami menambahkan aturan struktural yang tidak perlu dipelajari (seperti "baris pertama bukan nama" atau "satu baris tidak boleh dipakai dua kali"), hasilnya hampir selalu membantu.

Penjelasannya masuk akal. Jumlah contoh training untuk tiap field itu sedikit, mulai dari puluhan sampai beberapa ratus saja. Untuk fitur yang perlu dipelajari lewat data, jumlah contoh sekecil itu tidak cukup supaya model bisa generalisasi dengan baik. Sedangkan aturan struktural tidak butuh belajar dari data sama sekali, jadi tidak terpengaruh masalah data sedikit ini.

In [10]:
failed_experiments = pd.DataFrame([
    {"ide": "Kamus kata sambung nama (bin, binti, a-l, dst) sebagai fitur",
     "kenapa gagal": "Terlalu jarang muncul di data training, koefisien yang dipelajari model malah salah arah"},
    {"ide": "Skor frekuensi kata umum (boilerplate) sebagai fitur numerik biasa",
     "kenapa gagal": "Sama seperti di atas, model kesulitan belajar bobot yang tepat dari data sedikit"},
    {"ide": "Skor kemiripan huruf (character n-gram) dengan nama",
     "kenapa gagal": "Kena masalah yang sama, statistik huruf dihitung dari contoh yang terlalu sedikit dan beragam bahasa"},
    {"ide": "Fitur jarak ke wajah yang terdeteksi di foto",
     "kenapa gagal": "Tidak membantu maupun merugikan, informasinya sudah tercakup fitur posisi yang ada"},
    {"ide": "Voting antar 5 model classifier berbeda",
     "kenapa gagal": "Alamat jadi lebih buruk karena kalah suara oleh model model pohon yang memang lemah di alamat"},
    {"ide": "Gabungkan hasil OCR dua engine berbeda untuk nama dan alamat",
     "kenapa gagal": "Menambah baris teks membuat fitur posisi relatif jadi bergeser dan bikin salah pilih"},
    {"ide": "Preprocessing gambar (grayscale, koreksi perspektif, deteksi dokumen otomatis)",
     "kenapa gagal": "Rata rata netral atau malah sedikit merugikan, karena OCR memang sudah cukup baik sejak awal"},
])
failed_experiments

,ide,kenapa gagal
0,"Kamus kata sambung nama (bin, binti, a-l, dst)...","Terlalu jarang muncul di data training, koefis..."
1,Skor frekuensi kata umum (boilerplate) sebagai...,"Sama seperti di atas, model kesulitan belajar ..."
2,Skor kemiripan huruf (character n-gram) dengan...,"Kena masalah yang sama, statistik huruf dihitu..."
3,Fitur jarak ke wajah yang terdeteksi di foto,"Tidak membantu maupun merugikan, informasinya ..."
4,Voting antar 5 model classifier berbeda,Alamat jadi lebih buruk karena kalah suara ole...
5,Gabungkan hasil OCR dua engine berbeda untuk n...,Menambah baris teks membuat fitur posisi relat...
6,"Preprocessing gambar (grayscale, koreksi persp...","Rata rata netral atau malah sedikit merugikan,..."


## 10. Studi Kasus: Membedah Masalah pada Field Alamat

Field alamat paling sulit dibanding dua field lain, jadi kami luangkan waktu khusus untuk membedahnya lebih dalam. Caranya sederhana, kami urutkan semua prediksi alamat berdasarkan CER, lalu kami baca satu per satu contoh yang paling buruk dan contoh yang di tengah tengah (median), untuk mencari pola yang berulang.

Dari situ kami temukan beberapa pola kesalahan yang berulang dan bisa diperbaiki.

Field jenis kelamin dan agama (kata "LELAKI" untuk laki laki dan "ISLAM") kadang ikut tercampur ke prediksi alamat, karena posisinya di kartu memang berdekatan dengan alamat. Kami tambahkan aturan supaya kata kata ini tidak pernah dianggap bagian dari alamat.

Kadang OCR salah baca satu huruf dari kata kata di atas (misalnya "ISLAM" terbaca "SLAM"), jadi aturan pencocokan kata perlu dibuat toleran terhadap kesalahan kecil seperti ini, bukan cocok persis huruf per huruf saja.

Potongan nama orang (seperti "BIN IDRIS") kadang masih lolos tercampur ke alamat, terutama kalau potongan itu pendek sehingga skor "nama" nya dari model tidak terlalu tinggi. Kami tambahkan aturan tambahan, baris yang diawali kata seperti "BIN" atau "BINTI" tidak pernah dianggap alamat.

Ada juga pola baru yang kami temukan, yaitu potongan nomor identitas (angka saja, 6 digit atau lebih) yang tercetak dua kali di kartu (sekali lengkap, sekali terpotong sebagai stempel verifikasi) ikut tercampur ke alamat. Kami tambahkan aturan, blok teks yang isinya angka semua dan panjang, tidak pernah dianggap alamat, karena kami cek dulu tidak ada satu pun alamat asli di seluruh dataset yang isinya angka polos sepanjang itu.

Setiap perbaikan ini diuji satu per satu dan hasilnya konsisten membaik, tidak ada yang merugikan field lain.

In [11]:
# batas waktu dipakai untuk menandai kapan studi kasus alamat ini mulai
# dikerjakan (dilihat dari waktu asli di log eksperimen), bukan angka indeks
# yang gampang berubah kalau notebook ini dijalankan ulang di masa depan
STUDI_KASUS_ALAMAT_MULAI = pd.Timestamp("2026-08-19T18:00:00+00:00")

df_addr = exp[exp["method"] == "baseline1-line-classifier-geo"].copy()
df_addr["timestamp"] = pd.to_datetime(df_addr["timestamp"])
df_addr_val = df_addr[df_addr["split"] == "val"].sort_values("timestamp")

before_rows = df_addr_val[df_addr_val["timestamp"] < STUDI_KASUS_ALAMAT_MULAI]
after_rows = df_addr_val[df_addr_val["timestamp"] >= STUDI_KASUS_ALAMAT_MULAI]
before = before_rows.iloc[-1]["metrics"]["fields"]["address"]["character_error_rate"] * 100
after = after_rows.iloc[-1]["metrics"]["fields"]["address"]["character_error_rate"] * 100
print(f"CER alamat sebelum studi kasus ini dimulai : {before:.1f}%")
print(f"CER alamat setelah semua perbaikan di atas  : {after:.1f}%")

CER alamat sebelum studi kasus ini dimulai : 31.5%
CER alamat setelah semua perbaikan di atas  : 27.1%


## 11. Hasil Akhir Model

Berikut ringkasan performa model akhir kami, yaitu classifier logistic regression dengan seluruh lapisan perbaikan struktural yang sudah dijelaskan di atas. Ini adalah model yang kami simpan sebagai model produksi (`reports/line_classifier.pkl`).

Perlu saya sampaikan dengan jujur, field tanggal lahir sudah bisa dibilang selesai dengan baik, CER-nya di bawah 10 persen. Tapi field nama dan alamat masih di kisaran 20 sampai 30 persen CER. Ini bukan angka sempurna, tapi ini kemajuan nyata dari titik awal kami, di mana CER nama sempat di kisaran 50 persen dan alamat bahkan lebih tinggi lagi.

In [12]:
final_rows = []
for split in ("val", "test"):
    d = exp[(exp["method"] == "baseline1-line-classifier-geo") & (exp["split"] == split)].copy()
    d["timestamp"] = pd.to_datetime(d["timestamp"])
    r = d.sort_values("timestamp").iloc[-1]
    f = r["metrics"]["fields"]
    for field in ("name", "birth_date", "address"):
        final_rows.append({
            "split": split,
            "field": field,
            "exact_match_%": f[field]["exact_accuracy"] * 100,
            "CER_%": f[field]["character_error_rate"] * 100,
            "WER_%": f[field]["word_error_rate"] * 100,
        })
final_table = pd.DataFrame(final_rows).set_index(["split", "field"])
final_table.round(1)

exact_match_%  CER_%  WER_%
split field                                  
val   name                 38.8   20.6   42.0
      birth_date           89.8    6.9    7.5
      address               6.1   27.1   54.9
test  name                 31.2   29.0   43.5
      birth_date           91.7    8.3    8.3
      address               2.1   28.5   57.3

## 12. Seberapa Umum Perbaikan Perbaikan Ini Bisa Dipakai

Ini bagian yang menurut saya penting untuk jujur disampaikan. Tidak semua perbaikan yang kami buat punya tingkat generalisasi yang sama.

Beberapa aturan benar benar bersifat umum, artinya berlaku untuk dokumen apa pun dari negara mana pun, karena berdasarkan logika struktural murni, bukan kosakata bahasa tertentu. Contohnya aturan "baris pertama bukan nama", atau "satu baris tidak boleh dipakai untuk dua field sekaligus", atau aturan "blok angka panjang bukan alamat".

Tapi ada juga perbaikan yang sifatnya spesifik untuk dokumen berbahasa Melayu, seperti daftar kata umum ("KAD PENGENALAN", "WARGANEGARA") atau daftar kata sambung nama ("BIN", "BINTI"). Perbaikan ini terbukti membantu, tapi kontribusinya besar karena dataset evaluasi kami memang didominasi dokumen Malaysia. Kalau dipakai untuk dokumen dari negara lain yang tidak masuk daftar kosakata ini, perbaikan ini tidak akan memberi manfaat apa apa, meskipun juga tidak merugikan.

Jalan keluar yang sudah mulai kami bangun untuk masalah ini adalah menghitung kosakata "kata umum" secara otomatis dari data (seberapa sering satu teks muncul persis sama di banyak identitas berbeda), bukan mengetik manual satu per satu. Cara ini sudah kami uji dan terbukti benar mengenali kata umum bahasa Melayu tanpa perlu diberi tahu manual. Masalahnya, cara otomatis ini baru bisa mengenali kosakata bahasa lain kalau data training punya cukup banyak contoh dari bahasa itu, dan saat ini data kami belum cukup beragam untuk itu. Jadi ini bukan jalan buntu, cuma perlu lebih banyak data beragam ke depannya.

## 13. Keterbatasan dan Rencana Lanjutan

Terakhir, saya mau tulis dengan jujur apa saja yang belum selesai, supaya siapa pun yang melanjutkan pekerjaan ini tahu harus mulai dari mana.

Jumlah contoh training untuk field nama dan alamat masih sedikit dibanding keragaman layout dokumen yang ada. Ini adalah batasan paling mendasar yang kami temukan berulang kali sepanjang proses kerja.

Held out set yang berisi 100 gambar tanpa label belum pernah dipakai untuk pengecekan akhir. Idealnya, sebagian kecil dari situ dilabeli manual (ini diizinkan panduan proyek khusus untuk keperluan evaluasi, bukan training) untuk mengecek apakah model kami benar benar general atau cuma cocok dengan data yang sudah kami lihat berulang kali selama proses kerja ini.

Untuk field alamat, masih ada beberapa kasus kesalahan yang polanya tidak jelas, terutama nama orang tanpa kata sambung yang ikut tercampur ke alamat. Kami belum menemukan cara memperbaiki ini tanpa risiko merugikan bagian lain.

Kalau dataset ke depannya bertambah beragam, terutama untuk dokumen dari negara selain Malaysia, ada peluang besar CER untuk field nama dan alamat bisa turun lebih jauh lagi, karena sebagian besar mekanisme yang kami bangun memang dirancang untuk otomatis ikut membaik seiring data bertambah, bukan cuma bekerja untuk data yang ada sekarang saja.